# Notebook 3: Segmentação de Pessoas
Neste notebook, vamos implementar segmentação semântica para identificar pessoas em imagens.
Usaremos uma arquitetura DeepLabV3+ com backbone ResNet50.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet50
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pycocotools import mask as mask_utils

from core.dataset import COCOPersonDataset

In [ ]:
# Configurações
BATCH_SIZE = 4  # Menor devido ao tamanho das máscaras
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_SIZE = 512  # Tamanho fixo para treino

In [ ]:
class PersonSegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, coco_data, transform=None, mask_transform=None):
        self.coco = coco_data['coco']
        self.image_dir = coco_data['image_dir']
        self.image_ids = coco_data['image_ids']
        self.transform = transform
        self.mask_transform = mask_transform
        
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        # Carregar imagem
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        image = Image.open(os.path.join(self.image_dir, img_info['file_name']))
        
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        # Criar máscara binária para pessoas
        ann_ids = self.coco.getAnnIds(imgIds=img_id, catIds=[1])  # catId 1 = person
        anns = self.coco.loadAnns(ann_ids)
        
        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
        for ann in anns:
            if 'segmentation' in ann:
                if isinstance(ann['segmentation'], dict):
                    rle = ann['segmentation']
                    mask_ann = mask_utils.decode(rle)
                else:
                    # Convert polygon to mask
                    mask_ann = self.coco.annToMask(ann)
                mask = np.maximum(mask, mask_ann)
        
        mask = Image.fromarray(mask)
        
        # Aplicar transformações
        if self.transform is not None:
            image = self.transform(image)
        if self.mask_transform is not None:
            mask = self.mask_transform(mask)
        
        return image, mask.long()

In [ ]:
class PersonSegmenter(nn.Module):
    def __init__(self, num_classes=2):  # 2 classes: background e person
        super(PersonSegmenter, self).__init__()
        self.model = deeplabv3_resnet50(pretrained=True)
        self.model.classifier[-1] = nn.Conv2d(256, num_classes, kernel_size=1)
    
    def forward(self, x):
        return self.model(x)['out']

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct_pixels = 0
    total_pixels = 0
    
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calcular acurácia de pixels
        preds = outputs.argmax(1)
        correct_pixels += (preds == masks).sum().item()
        total_pixels += torch.numel(masks)
    
    avg_loss = total_loss / len(loader)
    pixel_acc = 100.0 * correct_pixels / total_pixels
    return avg_loss, pixel_acc

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct_pixels = 0
    total_pixels = 0
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            total_loss += loss.item()
            
            # Calcular acurácia de pixels
            preds = outputs.argmax(1)
            correct_pixels += (preds == masks).sum().item()
            total_pixels += torch.numel(masks)
    
    avg_loss = total_loss / len(loader)
    pixel_acc = 100.0 * correct_pixels / total_pixels
    return avg_loss, pixel_acc

In [ ]:
def visualize_segmentation(image, mask, pred_mask=None):
    # Converter tensor para numpy
    image = image.permute(1, 2, 0).cpu().numpy()
    image = (image * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406])
    image = np.clip(image, 0, 1)
    
    mask = mask.cpu().numpy()
    
    fig, axes = plt.subplots(1, 3 if pred_mask is not None else 2, figsize=(12, 4))
    
    # Imagem original
    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Máscara verdadeira
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title('Ground Truth Mask')
    axes[1].axis('off')
    
    if pred_mask is not None:
        # Máscara predita
        pred_mask = pred_mask.cpu().numpy()
        axes[2].imshow(pred_mask, cmap='gray')
        axes[2].set_title('Predicted Mask')
        axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Preparar transformações
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

mask_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

In [ ]:
# Preparar dataset
dataset = COCOPersonDataset()
person_data = dataset.get_person_data()

# Criar datasets
train_dataset = PersonSegmentationDataset(
    person_data,
    transform=transform,
    mask_transform=mask_transform
)

# Split train/val
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

# Criar dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

In [ ]:
# Inicializar modelo, critério e otimizador
model = PersonSegmenter().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# Arrays para acompanhar o progresso
train_losses = []
train_accs = []
val_losses = []
val_accs = []

# Loop de treino
for epoch in range(NUM_EPOCHS):
    # Treino
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Validação
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}]')
    print(f'Train Loss: {train_loss:.4f}, Train Pixel Acc: {train_acc:.2f}%')
    print(f'Val Loss: {val_loss:.4f}, Val Pixel Acc: {val_acc:.2f}%')
    print('-' * 50)

In [ ]:
# Plotar resultados
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Loss over epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Pixel Acc')
plt.plot(val_accs, label='Val Pixel Acc')
plt.title('Pixel Accuracy over epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Salvar modelo
torch.save(model.state_dict(), 'person_segmenter.pth')

In [ ]:
# Testar em algumas imagens
model.eval()
with torch.no_grad():
    for images, masks in list(val_loader)[:5]:  # visualizar 5 imagens
        images = images.to(DEVICE)
        outputs = model(images)
        pred_masks = outputs.argmax(1)
        
        # Visualizar primeira imagem do batch
        visualize_segmentation(images[0], masks[0], pred_masks[0])